# Small Sample Data Generator

Generate small BTC OHLCV samples for foundation building and testing.

**Purpose**: Create realistic 15-minute BTC data following NeuralForecast canonical format

**Author**: Configuration Architect (config-architect)

**Last Updated**: 2025-01-15

In [ ]:
# MANDATORY CONSTRAINTS FOR FOUNDATION BUILDING
SAMPLE_SIZE = 100  # MAX 1000 for testing
MAX_STEPS = 100    # Full training uses 20000
N_WINDOWS = 2      # Full CV uses 6-10
BATCH_SIZE = 32    # Full training uses 512

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pathlib import Path
import yaml
import ipywidgets as widgets
from IPython.display import display, HTML
import sys

# Add project root to path
PROJECT_ROOT = Path('/Users/mac-main/Neural-Forecast')
sys.path.append(str(PROJECT_ROOT))

# Import validation utilities
from utils.validate import (
    assert_regular_grid,
    assert_utc_eob,
    assert_shifted,
    assert_no_forward_fill_y
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Sample size: {SAMPLE_SIZE} rows")
print(f"Foundation mode active: Limited resources for testing")

## 1. Data Generation Parameters

Define realistic parameters for BTC OHLCV data generation

In [ ]:
# BTC realistic price ranges (as of 2024-2025)
BTC_PARAMS = {
    'base_price': 45000,  # Base BTC price in USD
    'volatility': 0.02,   # 2% daily volatility
    'trend': 0.0001,      # Slight upward trend
    'volume_base': 1000,  # Base volume in BTC
    'volume_volatility': 0.3,  # Volume volatility
}

# Time parameters
TIME_PARAMS = {
    'freq': '15min',
    'timezone': 'UTC',
    'start_date': '2024-01-01 00:00:00',  # UTC start
}

# Feature parameters for mock indicators
FEATURE_PARAMS = {
    'n_indicators': 20,  # Number of mock technical indicators
    'indicator_lag': 1,  # Shift by 1 to prevent leakage
    'time_features': ['minute_of_day', 'hour_of_day', 'day_of_week'],
}

print("Generation parameters defined:")
print(f"- Base BTC price: ${BTC_PARAMS['base_price']:,.0f}")
print(f"- Daily volatility: {BTC_PARAMS['volatility']*100:.1f}%")
print(f"- Frequency: {TIME_PARAMS['freq']}")
print(f"- Mock indicators: {FEATURE_PARAMS['n_indicators']}")

## 2. Generate Canonical Frame

Create NeuralForecast-compatible data following Section 2.3 (lines 751-810)

In [ ]:
def generate_btc_ohlcv(n_bars: int = 100, seed: int = 1337) -> pd.DataFrame:
    """Generate realistic BTC OHLCV data.
    
    Args:
        n_bars: Number of 15-minute bars to generate
        seed: Random seed for reproducibility
        
    Returns:
        DataFrame with OHLCV columns
    """
    np.random.seed(seed)
    
    # Generate timestamps (UTC, end-of-bar)
    start = pd.Timestamp(TIME_PARAMS['start_date'], tz='UTC')
    timestamps = pd.date_range(
        start=start,
        periods=n_bars,
        freq=TIME_PARAMS['freq'],
        tz='UTC'
    )
    
    # Generate price series using geometric Brownian motion
    returns = np.random.normal(
        loc=BTC_PARAMS['trend'],
        scale=BTC_PARAMS['volatility'] / np.sqrt(96),  # 96 bars per day
        size=n_bars
    )
    
    # Cumulative price series
    price_series = BTC_PARAMS['base_price'] * np.exp(np.cumsum(returns))
    
    # Generate OHLC from price series
    ohlcv_data = []
    for i in range(n_bars):
        base_price = price_series[i]
        
        # Intrabar volatility
        intrabar_vol = BTC_PARAMS['volatility'] / np.sqrt(96) * 0.5
        
        # Generate OHLC
        open_price = base_price * (1 + np.random.normal(0, intrabar_vol))
        high_price = base_price * (1 + abs(np.random.normal(0, intrabar_vol * 2)))
        low_price = base_price * (1 - abs(np.random.normal(0, intrabar_vol * 2)))
        close_price = base_price
        
        # Ensure OHLC relationships
        high_price = max(open_price, close_price, high_price)
        low_price = min(open_price, close_price, low_price)
        
        # Generate volume
        volume = BTC_PARAMS['volume_base'] * (1 + np.random.normal(0, BTC_PARAMS['volume_volatility']))
        volume = max(volume, 1)  # Ensure positive volume
        
        ohlcv_data.append({
            'open': open_price,
            'high': high_price,
            'low': low_price,
            'close': close_price,
            'volume': volume
        })
    
    # Create DataFrame
    df = pd.DataFrame(ohlcv_data, index=timestamps)
    df.index.name = 'ds'
    df = df.reset_index()
    
    return df


def create_canonical_frame(df: pd.DataFrame, unique_id: str = "BTC") -> pd.DataFrame:
    """Convert OHLCV data to NeuralForecast canonical format.
    
    Args:
        df: DataFrame with OHLCV data and 'ds' timestamp column
        unique_id: Identifier for the time series
        
    Returns:
        DataFrame in canonical format with unique_id, ds, y columns
    """
    # Calculate log returns as target
    df['y'] = np.log(df['close'] / df['close'].shift(1))
    
    # Create canonical frame
    canonical = pd.DataFrame({
        'unique_id': unique_id,
        'ds': df['ds'],
        'y': df['y']
    })
    
    # Add OHLCV as additional columns (for feature engineering)
    for col in ['open', 'high', 'low', 'close', 'volume']:
        canonical[col] = df[col]
    
    # Remove first row (NaN from returns calculation)
    canonical = canonical.iloc[1:].reset_index(drop=True)
    
    return canonical


# Generate sample data
print("Generating BTC OHLCV data...")
ohlcv_df = generate_btc_ohlcv(n_bars=SAMPLE_SIZE + 1, seed=1337)  # +1 for returns calculation
canonical_df = create_canonical_frame(ohlcv_df)

print(f"\n✅ Generated {len(canonical_df)} bars of BTC data")
print(f"\nFirst 5 rows:")
display(canonical_df.head())

print(f"\nData statistics:")
display(canonical_df[['y', 'close', 'volume']].describe())

## 3. Generate Feature Data

Add mock technical indicators and time-based features

In [ ]:
def add_mock_indicators(df: pd.DataFrame, n_indicators: int = 20, seed: int = 1337) -> pd.DataFrame:
    """Add mock technical indicators to the DataFrame.
    
    Args:
        df: Canonical DataFrame with OHLCV data
        n_indicators: Number of mock indicators to generate
        seed: Random seed
        
    Returns:
        DataFrame with added indicator columns
    """
    np.random.seed(seed)
    df = df.copy()
    
    # Generate different types of mock indicators
    indicator_types = [
        ('sma', 5),    # Moving averages
        ('ema', 5),    # Exponential moving averages
        ('rsi', 3),    # Momentum indicators
        ('bb', 3),     # Volatility indicators
        ('macd', 2),   # Trend indicators
        ('vol', 2),    # Volume indicators
    ]
    
    indicator_cols = []
    
    for indicator_type, count in indicator_types:
        for i in range(min(count, n_indicators - len(indicator_cols))):
            col_name = f"{indicator_type}_{i+1}"
            
            if indicator_type in ['sma', 'ema']:
                # Moving average-like indicators
                window = np.random.choice([5, 10, 20, 50])
                noise = np.random.normal(0, 0.001, len(df))
                df[col_name] = df['close'].rolling(window=min(window, len(df))).mean() + noise
                
            elif indicator_type == 'rsi':
                # Oscillator between 0 and 100
                base = 50 + 30 * np.sin(np.arange(len(df)) * 0.1 + np.random.rand() * 2 * np.pi)
                noise = np.random.normal(0, 5, len(df))
                df[col_name] = np.clip(base + noise, 0, 100)
                
            elif indicator_type == 'bb':
                # Bollinger band-like (distance from mean)
                window = np.random.choice([20, 30])
                rolling_std = df['close'].rolling(window=min(window, len(df))).std()
                df[col_name] = rolling_std * np.random.normal(0, 1, len(df))
                
            elif indicator_type == 'macd':
                # MACD-like (difference of moving averages)
                fast = df['close'].ewm(span=12, adjust=False).mean()
                slow = df['close'].ewm(span=26, adjust=False).mean()
                df[col_name] = (fast - slow) + np.random.normal(0, 0.0001, len(df))
                
            elif indicator_type == 'vol':
                # Volume-based indicators
                df[col_name] = df['volume'].rolling(window=10).mean() * np.random.uniform(0.8, 1.2, len(df))
            
            indicator_cols.append(col_name)
            
            if len(indicator_cols) >= n_indicators:
                break
    
    # Fill NaN values from rolling operations
    df[indicator_cols] = df[indicator_cols].fillna(method='bfill').fillna(0)
    
    # Apply shift(1) to all indicators to prevent leakage
    for col in indicator_cols:
        df[f"{col}_shifted"] = df[col].shift(1)
    
    # Drop unshifted columns and rename
    df = df.drop(columns=indicator_cols)
    df = df.rename(columns={f"{col}_shifted": col for col in indicator_cols})
    
    print(f"✅ Added {len(indicator_cols)} mock indicators with shift(1)")
    print(f"Indicators: {indicator_cols[:5]} ...")
    
    return df, indicator_cols


def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add time-based features to the DataFrame.
    
    Args:
        df: DataFrame with 'ds' timestamp column
        
    Returns:
        DataFrame with added time features
    """
    df = df.copy()
    
    # Convert ds to datetime if needed
    ds = pd.to_datetime(df['ds'])
    
    # Add time features
    df['minute_of_day'] = ds.dt.hour * 60 + ds.dt.minute
    df['hour_of_day'] = ds.dt.hour
    df['day_of_week'] = ds.dt.dayofweek
    df['is_weekend'] = (ds.dt.dayofweek >= 5).astype(int)
    
    # Cyclic encoding for time features
    df['hour_sin'] = np.sin(2 * np.pi * df['hour_of_day'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour_of_day'] / 24)
    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    
    time_features = ['minute_of_day', 'hour_of_day', 'day_of_week', 'is_weekend',
                    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']
    
    print(f"✅ Added {len(time_features)} time-based features")
    print(f"Time features: {time_features}")
    
    return df, time_features


# Add features to the canonical frame
print("Adding mock technical indicators...")
featured_df, hist_cols = add_mock_indicators(canonical_df, n_indicators=FEATURE_PARAMS['n_indicators'])

print("\nAdding time-based features...")
featured_df, time_cols = add_time_features(featured_df)

# Combine feature lists
all_features = hist_cols + time_cols

print(f"\n📊 Total features: {len(all_features)}")
print(f"\nDataFrame shape: {featured_df.shape}")
print(f"\nFirst 3 rows with features:")
display(featured_df[['unique_id', 'ds', 'y'] + hist_cols[:3] + time_cols[:3]].head(3))

## 4. Data Validation

Validate the generated data using project validation functions

In [ ]:
def validate_generated_data(df: pd.DataFrame, hist_cols: list) -> bool:
    """Validate generated data against all quality checks.
    
    Args:
        df: DataFrame to validate
        hist_cols: List of historical feature columns
        
    Returns:
        True if all validations pass
    """
    print("Running validation checks...\n")
    all_valid = True
    
    # Check 1: Regular grid
    try:
        assert_regular_grid(df, freq="15min")
        print("✅ Regular 15-minute grid validated")
    except AssertionError as e:
        print(f"❌ Regular grid check failed: {e}")
        all_valid = False
    
    # Check 2: UTC end-of-bar timestamps
    try:
        assert_utc_eob(df, freq="15min")
        print("✅ UTC end-of-bar timestamps validated")
    except AssertionError as e:
        print(f"❌ UTC EOB check failed: {e}")
        all_valid = False
    
    # Check 3: Historical features are shifted
    try:
        # Skip first 2 rows due to NaN from shift operations
        df_to_check = df.iloc[2:].copy()
        assert_shifted(df_to_check, hist_cols)
        print("✅ Historical features properly shifted (no leakage)")
    except AssertionError as e:
        print(f"❌ Shift check failed: {e}")
        all_valid = False
    
    # Check 4: No forward-fill of target
    try:
        assert_no_forward_fill_y(df)
        print("✅ No forward-fill of target variable")
    except AssertionError as e:
        print(f"❌ Forward-fill check failed: {e}")
        all_valid = False
    
    # Additional checks
    print("\nAdditional checks:")
    
    # Check for NaN values
    nan_counts = df.isna().sum()
    if nan_counts.sum() > len(hist_cols):  # Allow NaN in first row from shift
        print(f"⚠️  Warning: {nan_counts.sum()} NaN values found")
    else:
        print("✅ NaN values within expected range")
    
    # Check data types
    if df['ds'].dtype == 'datetime64[ns, UTC]':
        print("✅ Timestamp column has correct dtype")
    else:
        print(f"⚠️  Warning: ds dtype is {df['ds'].dtype}, expected datetime64[ns, UTC]")
    
    # Check value ranges
    if df['y'].std() < 0.1:  # Log returns should have reasonable volatility
        print("✅ Target variable (log returns) has reasonable volatility")
    else:
        print(f"⚠️  Warning: High volatility in target: std={df['y'].std():.4f}")
    
    return all_valid


# Validate the generated data
print("="*50)
print("DATA VALIDATION")
print("="*50)

is_valid = validate_generated_data(featured_df, hist_cols)

if is_valid:
    print("\n🎉 All validation checks passed! Data is ready for use.")
else:
    print("\n⚠️  Some validation checks failed. Review the data generation process.")

## 5. Export Functions

Functions to save and load test data

In [ ]:
def save_small_sample(df: pd.DataFrame, 
                     hist_cols: list,
                     time_cols: list,
                     filename: str = "small_sample.parquet") -> Path:
    """Save the generated sample data with metadata.
    
    Args:
        df: DataFrame to save
        hist_cols: List of historical feature columns
        time_cols: List of time feature columns
        filename: Output filename
        
    Returns:
        Path to saved file
    """
    # Create data directory if it doesn't exist
    data_dir = PROJECT_ROOT / 'data' / 'samples'
    data_dir.mkdir(parents=True, exist_ok=True)
    
    # Save main data
    output_path = data_dir / filename
    df.to_parquet(output_path, index=False)
    
    # Save metadata
    metadata = {
        'n_rows': len(df),
        'freq': '15min',
        'hist_exog_list': hist_cols,
        'futr_exog_list': [],  # Empty for now
        'stat_exog_list': time_cols,
        'target': 'y',
        'unique_id': df['unique_id'].iloc[0] if 'unique_id' in df.columns else 'BTC',
        'date_range': {
            'start': str(df['ds'].min()),
            'end': str(df['ds'].max())
        },
        'generation_params': {
            'sample_size': SAMPLE_SIZE,
            'n_indicators': len(hist_cols),
            'n_time_features': len(time_cols),
            'seed': 1337
        }
    }
    
    metadata_path = output_path.with_suffix('.yaml')
    with open(metadata_path, 'w') as f:
        yaml.dump(metadata, f, default_flow_style=False)
    
    print(f"✅ Data saved to {output_path}")
    print(f"✅ Metadata saved to {metadata_path}")
    
    return output_path


def load_test_data(filename: str = "small_sample.parquet") -> tuple:
    """Load test data and metadata.
    
    Args:
        filename: Filename to load
        
    Returns:
        Tuple of (DataFrame, metadata dict)
    """
    data_dir = PROJECT_ROOT / 'data' / 'samples'
    data_path = data_dir / filename
    metadata_path = data_path.with_suffix('.yaml')
    
    if not data_path.exists():
        raise FileNotFoundError(f"Data file not found: {data_path}")
    
    # Load data
    df = pd.read_parquet(data_path)
    
    # Load metadata
    if metadata_path.exists():
        with open(metadata_path, 'r') as f:
            metadata = yaml.safe_load(f)
    else:
        metadata = {}
        print(f"⚠️  Warning: Metadata file not found: {metadata_path}")
    
    print(f"✅ Loaded {len(df)} rows from {data_path}")
    
    if metadata:
        print(f"\nMetadata:")
        print(f"- Frequency: {metadata.get('freq')}")
        print(f"- Date range: {metadata.get('date_range', {}).get('start')} to {metadata.get('date_range', {}).get('end')}")
        print(f"- Historical features: {len(metadata.get('hist_exog_list', []))}")
        print(f"- Time features: {len(metadata.get('stat_exog_list', []))}")
    
    return df, metadata


# Save the generated sample
print("Saving generated sample data...\n")
saved_path = save_small_sample(
    featured_df,
    hist_cols=hist_cols,
    time_cols=time_cols,
    filename="btc_small_sample.parquet"
)

# Test loading it back
print("\nTesting data loading...\n")
loaded_df, loaded_metadata = load_test_data("btc_small_sample.parquet")

print("\n✅ Data export/import functions working correctly!")

## 6. Interactive Data Generator

Widget-based interface for generating custom samples

In [ ]:
class InteractiveSampleGenerator:
    """Interactive sample data generator with widgets."""
    
    def __init__(self):
        self.df = None
        self.hist_cols = []
        self.time_cols = []
        self.setup_widgets()
    
    def setup_widgets(self):
        """Create interactive widgets."""
        # Data size
        self.n_bars_widget = widgets.IntSlider(
            value=100, min=50, max=1000, step=50,
            description='Sample Size:',
            style={'description_width': 'initial'}
        )
        
        # Price parameters
        self.base_price_widget = widgets.IntSlider(
            value=45000, min=20000, max=100000, step=5000,
            description='Base BTC Price:',
            style={'description_width': 'initial'}
        )
        
        self.volatility_widget = widgets.FloatSlider(
            value=0.02, min=0.005, max=0.05, step=0.005,
            description='Daily Volatility:',
            style={'description_width': 'initial'},
            readout_format='.1%'
        )
        
        # Feature parameters
        self.n_indicators_widget = widgets.IntSlider(
            value=20, min=5, max=50, step=5,
            description='N Indicators:',
            style={'description_width': 'initial'}
        )
        
        self.add_time_features_widget = widgets.Checkbox(
            value=True,
            description='Add Time Features',
            style={'description_width': 'initial'}
        )
        
        # Seed
        self.seed_widget = widgets.IntText(
            value=1337,
            description='Random Seed:',
            style={'description_width': 'initial'}
        )
        
        # Generate button
        self.generate_button = widgets.Button(
            description='Generate Data',
            button_style='primary',
            icon='play'
        )
        self.generate_button.on_click(self.generate_data)
        
        # Save button
        self.save_button = widgets.Button(
            description='Save Data',
            button_style='success',
            icon='save',
            disabled=True
        )
        self.save_button.on_click(self.save_data)
        
        # Output area
        self.output = widgets.Output()
    
    def generate_data(self, b=None):
        """Generate data based on widget values."""
        with self.output:
            from IPython.display import clear_output
            clear_output()
            
            print("Generating data...\n")
            
            # Update BTC parameters
            BTC_PARAMS['base_price'] = self.base_price_widget.value
            BTC_PARAMS['volatility'] = self.volatility_widget.value
            
            # Generate OHLCV
            ohlcv = generate_btc_ohlcv(
                n_bars=self.n_bars_widget.value + 1,
                seed=self.seed_widget.value
            )
            
            # Create canonical frame
            self.df = create_canonical_frame(ohlcv)
            
            # Add indicators
            self.df, self.hist_cols = add_mock_indicators(
                self.df,
                n_indicators=self.n_indicators_widget.value,
                seed=self.seed_widget.value
            )
            
            # Add time features if requested
            if self.add_time_features_widget.value:
                self.df, self.time_cols = add_time_features(self.df)
            else:
                self.time_cols = []
            
            # Validate
            print("\n" + "="*50)
            print("VALIDATION")
            print("="*50)
            is_valid = validate_generated_data(self.df, self.hist_cols)
            
            # Show summary
            print("\n" + "="*50)
            print("DATA SUMMARY")
            print("="*50)
            print(f"Shape: {self.df.shape}")
            print(f"Historical features: {len(self.hist_cols)}")
            print(f"Time features: {len(self.time_cols)}")
            print(f"\nFirst 3 rows:")
            display(self.df[['unique_id', 'ds', 'y', 'close']].head(3))
            
            # Enable save button
            self.save_button.disabled = False
            
            if is_valid:
                print("\n✅ Data is ready to save!")
            else:
                print("\n⚠️  Data has validation issues but can still be saved.")
    
    def save_data(self, b=None):
        """Save the generated data."""
        with self.output:
            if self.df is None:
                print("❌ No data to save. Generate data first.")
                return
            
            filename = f"btc_sample_{self.n_bars_widget.value}_bars.parquet"
            path = save_small_sample(
                self.df,
                hist_cols=self.hist_cols,
                time_cols=self.time_cols,
                filename=filename
            )
            print(f"\n✅ Data saved as {filename}")
    
    def display(self):
        """Display the interactive generator."""
        # Layout widgets
        data_section = widgets.VBox([
            widgets.HTML("<h3>Data Parameters</h3>"),
            self.n_bars_widget,
            self.seed_widget,
        ])
        
        price_section = widgets.VBox([
            widgets.HTML("<h3>Price Parameters</h3>"),
            self.base_price_widget,
            self.volatility_widget,
        ])
        
        feature_section = widgets.VBox([
            widgets.HTML("<h3>Feature Parameters</h3>"),
            self.n_indicators_widget,
            self.add_time_features_widget,
        ])
        
        button_section = widgets.HBox([
            self.generate_button,
            self.save_button
        ])
        
        main_layout = widgets.HBox([data_section, price_section, feature_section])
        
        display(widgets.VBox([
            widgets.HTML("<h2>Interactive Sample Generator</h2>"),
            main_layout,
            button_section,
            self.output
        ]))

# Create and display the generator
generator = InteractiveSampleGenerator()
generator.display()

## Summary

This notebook provides:
1. ✅ Realistic BTC OHLCV data generation at 15-minute frequency
2. ✅ NeuralForecast canonical frame format (unique_id, ds, y)
3. ✅ Mock technical indicators with proper shift(1) to prevent leakage
4. ✅ Time-based features for temporal patterns
5. ✅ Full validation using project validation utilities
6. ✅ Export/import functions for data persistence
7. ✅ Interactive widget-based generator for custom samples

The generated data is suitable for:
- Testing model instantiation
- Validating feature engineering pipelines
- Running small-scale cross-validation
- Debugging training workflows

All data passes the required validation checks:
- Regular 15-minute grid
- UTC end-of-bar timestamps
- Properly shifted historical features
- No forward-fill of target variable